# FlipAware pipeline

One notebook for **Google Colab** and **RunPod** (Jupyter). It detects where it runs:

| | Colab | RunPod |
|---|---|---|
| Profile | `small` (quick test) | `full` (real release) |
| Code folder | `/content/flipaware` | `/workspace/flipaware` |
| Output folder | Google Drive `flipaware_work` | `/workspace/flipaware_work` |
| Token | Colab secret `HF_TOKEN` | environment variable `HF_TOKEN`, or typed once (hidden) |

**Never paste a token into a cell.** On Colab use the key icon (Secrets). On RunPod set `HF_TOKEN` in the pod's environment variables, or type it when cell 2 asks.

If the session drops, run cells 1 to 4 again and continue from the stage you were on. Finished work is skipped, and a stage reruns by itself when its inputs changed.

## 1. Settings

In [ ]:
PROFILE = "auto"
TRACKERS = "tensorboard,mlflow"
MODELS = ""

## 2. Platform, GPU, token

In [ ]:
import os, subprocess, getpass
ON_COLAB = os.path.isdir("/content") and os.path.isdir("/content/sample_data") or "COLAB_RELEASE_TAG" in os.environ
ON_RUNPOD = bool(os.environ.get("RUNPOD_POD_ID"))
print("platform:", "colab" if ON_COLAB else "runpod" if ON_RUNPOD else "other")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

if ON_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:
        os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
    except Exception:
        pass
elif not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace write token (hidden): ")

from huggingface_hub import HfApi
print("logged in as", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])

if PROFILE != "auto":
    os.environ["FLIP_PROFILE"] = PROFILE
os.environ["FLIP_TRACKERS"] = TRACKERS
if MODELS:
    os.environ["FLIP_MODELS"] = MODELS
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

## 3. Code and packages

In [ ]:
REPO = "/content/flipaware" if ON_COLAB else "/workspace/flipaware" if os.path.isdir("/workspace") else os.path.expanduser("~/flipaware")
if not os.path.isdir(REPO):
    url = "https://github.com/amareshhebbar/flipaware.git"
    if os.environ.get("GH_TOKEN"):
        url = f"https://{os.environ['GH_TOKEN']}@github.com/amareshhebbar/flipaware.git"
    subprocess.run(["git", "clone", url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
r = subprocess.run(["bash", "setup.sh"], capture_output=True, text=True)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "dependency resolver" not in l and "requires" not in l)[-1500:])
WORK = subprocess.run(["python", "-m", "flipaware.common", "workdir"], capture_output=True, text=True).stdout.strip()
os.makedirs(f"{WORK}/logs", exist_ok=True)
GEN_PY = "/workspace/.venv_gen/bin/python" if os.path.exists("/workspace/.venv_gen/bin/python") else "python"
print("code:", REPO, "\nwork:", WORK, "\ngeneration python:", GEN_PY)

## 4. Helpers
`stage()` runs one stage live and saves its log. `reset(n)` deletes outputs of stage n and later, for a clean rerun.

In [ ]:
import sys, json
import pandas as pd
from IPython.display import display, Image, Markdown
pd.set_option("display.max_colwidth", 160)

def stage(n, mod, *args):
    tag = f"{n:02d}_{mod}" + ("_" + "".join(c for c in "".join(args) if c.isalpha()) if args else "")
    py = GEN_PY if mod == "s02_generate" else sys.executable
    with open(f"{WORK}/logs/{tag}.log", "a") as log:
        p = subprocess.Popen([py, "-m", f"flipaware.{mod}", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
        for line in p.stdout:
            if "UserWarning" in line or "from pandas.core import" in line:
                continue
            print(line, end="")
            log.write(line)
        p.wait()
    print(f"\n=== stage {n} {mod} {' '.join(args)} finished with code {p.returncode}")
    if p.returncode != 0:
        raise RuntimeError(f"stage {n} failed, read the output above")

def reset(n):
    subprocess.run([sys.executable, "-m", "flipaware.reset", str(n), "--yes"], cwd=REPO)

def jl(name, folder="data"):
    p = f"{WORK}/{folder}/{name}"
    return pd.read_json(p, lines=True) if os.path.exists(p) and os.path.getsize(p) else pd.DataFrame()

def js(name, folder="results"):
    p = f"{WORK}/{folder}/{name}"
    return json.load(open(p)) if os.path.exists(p) else {}

def show(df, cols, n=6, seed=0):
    if df.empty:
        print("(nothing to show)")
        return
    display(df.sample(min(n, len(df)), random_state=seed)[[c for c in cols if c in df.columns]])
print("helpers ready")

## RunPod: run everything in the background (optional)
On RunPod the full run takes many hours. Running it in the background keeps it alive if you close the browser. Skip this on Colab and use the stage cells below instead.

In [ ]:
if ON_RUNPOD:
    subprocess.Popen(f"nohup bash run_all.sh > {WORK}/run.log 2>&1 &", shell=True, cwd=REPO)
    print("started, follow it with the next cell")
else:
    print("not on RunPod, use the stage cells below")

In [ ]:
print(subprocess.run(["tail", "-n", "40", f"{WORK}/run.log"], capture_output=True, text=True).stdout)
print(subprocess.run(f"grep -A12 'RESULT' {WORK}/run.log | tail -n 40", shell=True, capture_output=True, text=True).stdout)

## Stage 0: environment check

In [ ]:
stage(0, "s00_gpu_check")

## Stage 1: source data

In [ ]:
stage(1, "s01_data")
s = jl("seeds.jsonl")
if len(s):
    print(len(s), "seeds |", int(s.has_number.sum()), "with numbers |", int(s.context.notna().sum()), "with passages")
print(len(jl("human_anchors.jsonl")), "human anchors |", len(jl("nevir_test.jsonl")), "NevIR pairs")
show(s, ["text", "source", "has_number"], 5)

## Stage 2: generation
Watch the `pass the edit check` count in the log. Above 50% is healthy. If old generations exist from an earlier prompt, they are moved aside automatically.

In [ ]:
stage(2, "s02_generate", "--set", "main")
r = jl("raw.jsonl")
if len(r):
    print(len(r), "generations | attempts used:", r.get("attempts", pd.Series(dtype=float)).value_counts().sort_index().to_dict())
show(r.dropna(subset=["paraphrase"]) if "paraphrase" in r else r, ["type", "anchor", "paraphrase", "trap", "attempts"], 8, 1)

## Stage 3: filter
Stops with examples if fewer than 20 triples pass. Then read the rejected examples, adjust, run `reset(2)`, and redo stages 2 and 3.

In [ ]:
stage(3, "s03_filter", "--set", "main")

In [ ]:
st = pd.DataFrame(js("filter_stats.json")).T
if len(st):
    st["keep_%"] = (st.kept / st.generated * 100).round(1)
    display(st)
c = jl("candidates.jsonl")
if len(c):
    print(c.reason.value_counts().to_dict())
    print("kept examples"); show(c[c.kept], ["type", "anchor", "paraphrase", "trap", "nli_para", "nli_trap"], 6, 2)
    print("rejected examples"); show(c[~c.kept], ["type", "reason", "anchor", "paraphrase", "trap"], 6, 3)

## Stage 4: splits and passage traps

In [ ]:
stage(4, "s04_splits")
for f in ("train.jsonl", "dev.jsonl", "test.jsonl", "passage_test.jsonl"):
    print(f, len(jl(f)))
p = jl("passage_test.jsonl")
if len(p):
    x = p.iloc[0]
    print("\nquery:", x["query"], "\noriginal:", x["original"], "\ntwin:", x["twin"])

## Stages 5 and 6: cross generator test set

In [ ]:
stage(5, "s02_generate", "--set", "xgen")
stage(6, "s03_filter", "--set", "xgen")
x = jl("xgen_test.jsonl")
print(len(x), "cross generator triples")
show(x, ["type", "anchor", "paraphrase", "trap"], 5)

## Stage 7: upload the dataset

In [ ]:
stage(7, "s07_dataset")

## Stage 8: train, upload, test, verdict
Look for the `RESULT ... GOOD` or `BAD` block for each model.

In [ ]:
stage(8, "s08_models")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {WORK}/runs

## Stage 9: human test set
Skips until the writer Space has approved triples. That is expected.

In [ ]:
stage(9, "s09_human")

## Stage 10: analysis

In [ ]:
stage(10, "s10_analysis")

## Stage 11: report, cards, leaderboard

In [ ]:
stage(11, "s11_finalize")
if os.path.exists(f"{WORK}/results/summary.md"):
    display(Markdown(open(f"{WORK}/results/summary.md").read()))
for img in ("headline.png", "benchmarks.png", "dilution.png", "scaling.png", "blend.png", "leaderboard.png"):
    p = f"{WORK}/assets/{img}"
    if os.path.exists(p):
        print(img); display(Image(p, width=700))

In [ ]:
if os.path.exists(f"{WORK}/results/REPORT.md"):
    display(Markdown(open(f"{WORK}/results/REPORT.md").read()))

## Stage 12: publish logs and dashboards

In [ ]:
stage(12, "s12_logs")
try:
    import mlflow
    mlflow.set_tracking_uri(f"sqlite:///{WORK}/tracking/mlflow.db")
    runs = mlflow.search_runs(search_all_experiments=True)
    display(runs[[c for c in runs.columns if c in ("tags.mlflow.runName", "metrics.trap/accuracy", "metrics.nanobeir/ndcg10", "metrics.train_loss")]])
except Exception as e:
    print("mlflow view skipped:", e)

## Links to everything created

In [ ]:
api = HfApi(token=os.environ["HF_TOKEN"])
me = api.whoami()["name"]
suffix = "_smoke" if json.load(open(f"{WORK}/env.json")).get("profile") == "small" else ""
for kind, items, prefix in (("model", api.list_models(author=me, search="FlipAware"), ""), ("dataset", api.list_datasets(author=me, search="FlipAware"), "datasets/"), ("space", api.list_spaces(author=me, search="FlipAware"), "spaces/")):
    for it in items:
        if suffix and not it.id.endswith(suffix):
            continue
        print(f"{kind:8} https://huggingface.co/{prefix}{it.id}")
if js("collection.json"):
    print("collection https://huggingface.co/collections/" + js("collection.json")["slug"])

## Optional: human writer Space

In [ ]:
subprocess.run([sys.executable, "-m", "flipaware.human", "deploy"], cwd=REPO)

## Optional: debug bundle
Zips logs and results (no models). Send this file if something breaks.

In [ ]:
out = ("/content/drive/MyDrive" if ON_COLAB else WORK) + "/flipaware_debug.zip"
subprocess.run(f"cd {WORK} && zip -qr {out} logs results data/.stamp_* tracking/metrics.jsonl data/candidates.jsonl -x '*.db'", shell=True)
print(out)

## Clean rerun from a stage
Only when you want to redo work on purpose, for example `reset(2)` to regenerate the data.

In [ ]:
RESET_FROM = None
if RESET_FROM is not None:
    reset(RESET_FROM)

## Cleanup of the Colab test repos
Deletes every `_smoke` model, dataset, Space, and the smoke collection. Run only after checking them.

In [ ]:
CONFIRM = False
if CONFIRM:
    for m in api.list_models(author=me, search="_smoke"):
        api.delete_repo(m.id); print("deleted", m.id)
    for d in api.list_datasets(author=me, search="_smoke"):
        api.delete_repo(d.id, repo_type="dataset"); print("deleted", d.id)
    for s in api.list_spaces(author=me, search="_smoke"):
        api.delete_repo(s.id, repo_type="space"); print("deleted", s.id)
    for c in api.list_collections(owner=me):
        if c.title.endswith("smoke"):
            api.delete_collection(c.slug); print("deleted", c.slug)
else:
    print("set CONFIRM = True to delete")